In [8]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# ==============================================================================
# 1. CONFIGURAÇÃO DO ARQUIVO DE DADOS (SUPORTA 1 OU 2 MÃOS = 126 FEATURES)
# ==============================================================================
arquivo_csv = "meu_dataset_libras_completo.csv"

# 21 pontos * 3 eixos (x,y,z) * 2 mãos = 126 colunas (se usar 1 mão, a segunda fica zerada)
colunas = [f"m{m}_{eixo}_{i}" for m in [1, 2] for i in range(21) for eixo in ['x', 'y', 'z']]

if os.path.exists(arquivo_csv):
    df_existente = pd.read_csv(arquivo_csv)
    # Verifica se o CSV antigo tem as mesmas 126 colunas
    if len(df_existente.columns) - 1 == len(colunas):
        dados_coletados = df_existente.drop(columns=['classe']).values.tolist()
        labels_coletados = df_existente['classe'].astype(str).tolist()
        print(f"Dataset carregado! {len(labels_coletados)} amostras existentes.")
    else:
        dados_coletados, labels_coletados = [], []
else:
    dados_coletados, labels_coletados = [], []

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# max_num_hands=2 permite gravar sinais compostos como 'CASA', 'FAMILIA', etc.
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("\n" + "="*65)
print(" COMO USAR O COLETOR DE LETRAS E PALAVRAS:")
print(" • [TAB]        : Entra no modo de digitação (digite ex: CASA, AMOR, A)")
print(" • [ENTER]      : Confirma o nome do sinal digitado")
print(" • [ESPAÇO]     : Grava uma rajada de 50 frames para o sinal atual!")
print(" • [ESC]        : Finaliza e treina o modelo Random Forest")
print("="*65 + "\n")

sinal_atual = "CASA"       # Nome do sinal selecionado no momento
modo_digitacao = False     # Controla se o teclado está digitando um nome novo
texto_digitando = ""
contador_rajada = 0
TAMANHO_RAJADA = 50

# ==============================================================================
# 2. LAÇO DE CAPTURA E INTERFACE NA WEBCAM
# ==============================================================================
while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        break

    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resultado = hands.process(frame_rgb)

    vetor_frame = []

    if resultado.multi_hand_landmarks:
        # Ordena as mãos detectadas da esquerda para a direita na tela para manter consistência
        maos_ordenadas = sorted(resultado.multi_hand_landmarks, key=lambda h: h.landmark[0].x)
        
        for hand_landmarks in maos_ordenadas[:2]:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

            pulso_x = hand_landmarks.landmark[0].x
            pulso_y = hand_landmarks.landmark[0].y
            pulso_z = hand_landmarks.landmark[0].z

            for lm in hand_landmarks.landmark:
                vetor_frame.extend([lm.x - pulso_x, lm.y - pulso_y, lm.z - pulso_z])

        # Se apenas 1 mão estiver na tela, preenche os outros 63 valores da 2ª mão com 0.0
        if len(maos_ordenadas) == 1:
            vetor_frame.extend([0.0] * 63)

        # Gravação em rajada ativa
        if contador_rajada > 0:
            dados_coletados.append(vetor_frame)
            labels_coletados.append(sinal_atual)
            contador_rajada -= 1
            if contador_rajada == 0:
                print(f"Gravação de '{sinal_atual}' concluída! Total acumulado: {len(labels_coletados)}")

    # Painel visual no topo do vídeo
    cv2.rectangle(frame, (10, 10), (630, 115), (25, 25, 25), -1)

    if modo_digitacao:
        cv2.putText(frame, f"DIGITE O SINAL: {texto_digitando}_", (20, 45), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2)
        cv2.putText(frame, "Pressione [ENTER] para confirmar ou [BACKSPACE] para apagar", (20, 80), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    elif contador_rajada > 0:
        cv2.putText(frame, f"GRAVANDO '{sinal_atual}'... Faltam {contador_rajada} frames", (20, 45), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
    else:
        cv2.putText(frame, f"Sinal Ativo: [{sinal_atual}]", (20, 45), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
        cv2.putText(frame, "[ESPACO] Gravar 50x | [TAB] Mudar Nome | [ESC] Treinar", (20, 78), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1)

    classes_atuais = sorted(list(set(labels_coletados)))
    cv2.putText(frame, f"Amostras: {len(labels_coletados)} | Sinais: {', '.join(classes_atuais[-5:])}", 
                (20, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (180, 255, 180), 1)

    cv2.imshow("Coletor de Letras e Palavras - Libras", frame)

    tecla = cv2.waitKey(1) & 0xFF

    if tecla == 27:  # ESC sai e inicia o treinamento
        break

    if modo_digitacao:
        if tecla in [13, 10]:  # ENTER confirma a palavra digitada
            if len(texto_digitando.strip()) > 0:
                sinal_atual = texto_digitando.strip().upper()
            modo_digitacao = False
        elif tecla == 8:  # BACKSPACE apaga última letra
            texto_digitando = texto_digitando[:-1]
        elif 32 <= tecla <= 126:  # Letras, números ou espaço
            texto_digitando += chr(tecla).upper()
    else:
        if tecla == 9:  # Tecla TAB ativa modo de digitação
            modo_digitacao = True
            texto_digitando = ""
        elif tecla == 32:  # BARRA DE ESPAÇO inicia a gravação da rajada
            if len(vetor_frame) == 126 and contador_rajada == 0:
                contador_rajada = TAMANHO_RAJADA
                print(f"Gravando 50 amostras para '{sinal_atual}'...")
            elif len(vetor_frame) == 0:
                print("Aviso: Posicione a(s) mão(s) na câmera antes de apertar ESPAÇO!")

cap.release()
cv2.destroyAllWindows()
hands.close()

# ==============================================================================
# 3. TREINAMENTO DO RANDOM FOREST
# ==============================================================================
if len(set(labels_coletados)) < 2:
    print("\n[Atenção] Grave pelo menos 2 sinais diferentes para treinar o modelo!")
else:
    df = pd.DataFrame(dados_coletados, columns=colunas)
    df['classe'] = labels_coletados
    df.to_csv(arquivo_csv, index=False)

    X = df.drop(columns=['classe'])
    y = df['classe'].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print("\nTreinando Random Forest para letras e palavras...")
    clf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print(f"\nACURÁCIA ALCANÇADA: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print(classification_report(y_test, y_pred))

    dados_modelo = {
        "modelo": clf,
        "colunas": list(X.columns),
        "num_features": 126
    }
    joblib.dump(dados_modelo, "modelo_asl_rf.pkl")
    print("Modelo salvo em 'modelo_asl_rf.pkl'!")


 COMO USAR O COLETOR DE LETRAS E PALAVRAS:
 • [TAB]        : Entra no modo de digitação (digite ex: CASA, AMOR, A)
 • [ENTER]      : Confirma o nome do sinal digitado
 • [ESPAÇO]     : Grava uma rajada de 50 frames para o sinal atual!
 • [ESC]        : Finaliza e treina o modelo Random Forest

Aviso: Posicione a(s) mão(s) na câmera antes de apertar ESPAÇO!
Gravando 50 amostras para 'CASA'...
Gravação de 'CASA' concluída! Total acumulado: 50
Gravando 50 amostras para 'DESCULPA'...
Gravação de 'DESCULPA' concluída! Total acumulado: 100
Gravando 50 amostras para 'EM PE'...
Gravação de 'EM PE' concluída! Total acumulado: 150
Gravando 50 amostras para 'A'...
Gravação de 'A' concluída! Total acumulado: 200
Gravando 50 amostras para 'I'...
Gravação de 'I' concluída! Total acumulado: 250
Gravando 50 amostras para 'L'...
Gravação de 'L' concluída! Total acumulado: 300

Treinando Random Forest para letras e palavras...

ACURÁCIA ALCANÇADA: 100.00%
              precision    recall  f1-score   s